# 04 -- Websockets (client et serveur)

## Objectifs pedagogiques

A la fin de ce notebook, vous saurez :

- comprendre le protocole WebSocket ;
- creer un serveur WebSocket avec la bibliotheque `websockets` ;
- ecrire un client qui se connecte et echange des messages ;
- gerer les erreurs reseau et le reconnect ;
- implementer un broadcast multi-clients ;
- utiliser ping/pong et heartbeat ;
- echanger du JSON ;
- tester un handler avec `websockets.serve`.

## Prerequis -- ce que vous devez maitriser

- asyncio, async/await ;
- reseau TCP basique ;
- JSON.

Ce que ce notebook **ne couvrira pas** :

- les WebSockets dans FastAPI (voir `Web_backend/05_Websockets_realtime.ipynb`) ;
- Socket.IO, SSE ;
- WebRTC.

## Plan

1. Le protocole WebSocket
2. Installation
3. Serveur minimal
4. Client minimal
5. Echange de JSON
6. Broadcast
7. Gestion d'erreurs
8. Reconnexion automatique
9. Ping/Pong et heartbeat
10. TLS (wss://)
11. Synthese
12. Exercices

---

## 1. Le protocole WebSocket

- Base sur TCP
- Demarre par un handshake HTTP avec `Upgrade: websocket`
- Communication bidirectionnelle full-duplex
- Messages en frames (text ou binary)
- Code de fermeture standardise (1000 OK, 1008 violation, 1011 internal...)

---

## 2. Installation

In [ ]:
# pip install websockets
import websockets
print(websockets.__version__)

---

## 3. Serveur minimal

In [ ]:
import asyncio
import websockets

async def echo(websocket):
    async for message in websocket:
        await websocket.send(f'Echo: {message}')

async def main():
    async with websockets.serve(echo, 'localhost', 8765):
        await asyncio.Future()  # bloque

# asyncio.run(main())  # ne pas lancer dans notebook
print('serveur defini')

---

## 4. Client minimal

In [ ]:
async def client():
    async with websockets.connect('ws://localhost:8765') as ws:
        await ws.send('Hello')
        response = await ws.recv()
        return response

print('client defini')

---

## 5. Echange de JSON

In [ ]:
import json

async def json_server(ws):
    async for raw in ws:
        data = json.loads(raw)
        await ws.send(json.dumps({'received': data, 'len': len(raw)}))

async def json_client():
    async with websockets.connect('ws://localhost:8765') as ws:
        await ws.send(json.dumps({'hello': 'world'}))
        return json.loads(await ws.recv())

print('handlers JSON definis')

---

## 6. Broadcast

In [ ]:
CLIENTS: set = set()

async def broadcast_handler(ws):
    CLIENTS.add(ws)
    try:
        async for message in ws:
            await asyncio.gather(
                *(c.send(message) for c in CLIENTS if c is not ws),
                return_exceptions=True,
            )
    finally:
        CLIENTS.discard(ws)

print('broadcast handler defini')

---

## 7. Gestion d'erreurs

In [ ]:
from websockets.exceptions import ConnectionClosed, ConnectionClosedError

async def robust_handler(ws):
    try:
        async for message in ws:
            await ws.send(f'Ok: {message}')
    except ConnectionClosedError:
        print('Client parti brutalement')
    except ConnectionClosed:
        print('Client ferme proprement')

print('handler robuste')

---

## 8. Reconnexion automatique

In [ ]:
async def reconnect_client(url: str) -> None:
    backoff = 1
    while True:
        try:
            async with websockets.connect(url) as ws:
                backoff = 1
                async for message in ws:
                    print('recv :', message)
        except (ConnectionClosedError, OSError):
            await asyncio.sleep(backoff)
            backoff = min(backoff * 2, 60)

print('reconnect avec backoff exponentiel')

---

## 9. Ping/Pong et heartbeat

In [ ]:
# La lib gere ping/pong automatiquement
# websockets.serve(handler, 'localhost', 8765, ping_interval=20, ping_timeout=20)
# Cote client : websockets.connect(url, ping_interval=10, ping_timeout=10)
print('ping_interval et ping_timeout en secondes')

---

## 10. TLS (wss://)

In [ ]:
import ssl
ctx = ssl.create_default_context(ssl.Purpose.CLIENT_AUTH)
# ctx.load_cert_chain(certfile='cert.pem', keyfile='key.pem')

# Serveur TLS : websockets.serve(handler, 'localhost', 8765, ssl=ctx)
# Client TLS : websockets.connect('wss://example.com:8765')
print('wss = websocket over TLS')

---

## Synthese

| Operation | websockets |
|---|---|
| Serveur | `websockets.serve(handler, host, port)` |
| Client | `websockets.connect(url)` |
| Envoyer | `await ws.send(msg)` |
| Recevoir | `await ws.recv()` / `async for msg in ws:` |
| Fermer | `await ws.close(code, reason)` |
| Broadcast | `asyncio.gather(*(c.send(m) for c in clients))` |
| Heartbeat | `ping_interval`, `ping_timeout` |
| TLS | `ssl=ssl.create_default_context(...)` |

### Regles a retenir

- **`async for msg in ws`** : boucle naturelle tant que connecte.
- **`CLIENTS.discard`** pour nettoyer dans `finally`.
- **`asyncio.gather(..., return_exceptions=True)`** pour un broadcast robuste.
- **`wss://`** en production, jamais `ws://`.

---

## Exercices

### Exercice 1 (facile) -- Echo

Ecrivez un handler qui renvoie chaque message prefixe par `REP: `.

In [ ]:
import websockets

# TODO

<details>
<summary>Voir la correction</summary>

```python
async def echo(ws):
    async for msg in ws:
        await ws.send(f'REP: {msg}')
```

</details>

### Exercice 2 (facile) -- Client simple

Ecrivez une fonction async `send_and_recv(url, msg)` qui se connecte, envoie, recoit et ferme.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
import websockets

async def send_and_recv(url: str, msg: str) -> str:
    async with websockets.connect(url) as ws:
        await ws.send(msg)
        return await ws.recv()
```

</details>

### Exercice 3 (moyen) -- Chat broadcast

Handler de chat : chaque message est diffuse a tous les clients connectes.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
import asyncio
import websockets

clients: set = set()

async def chat(ws):
    clients.add(ws)
    try:
        async for msg in ws:
            await asyncio.gather(*(c.send(msg) for c in clients), return_exceptions=True)
    finally:
        clients.discard(ws)
```

</details>

### Exercice 4 (moyen) -- Serveur JSON calculatrice

Handler qui recoit `{"op": "add", "a": 1, "b": 2}` et renvoie `{"result": 3}`.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
import json

async def calc(ws):
    async for raw in ws:
        data = json.loads(raw)
        op, a, b = data['op'], data['a'], data['b']
        result = {'add': a + b, 'sub': a - b, 'mul': a * b, 'div': a / b}.get(op)
        await ws.send(json.dumps({'result': result}))
```

</details>

### Exercice 5 (difficile) -- Client qui reconnecte

Ecrivez un client async qui :
1. se connecte a `ws://localhost:8765` ;
2. envoie un ping toutes les 5 secondes ;
3. reconnecte avec backoff exponentiel si la connexion tombe.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
import asyncio
import websockets
from websockets.exceptions import ConnectionClosed

async def run_client(url: str) -> None:
    backoff = 1
    while True:
        try:
            async with websockets.connect(url) as ws:
                backoff = 1
                while True:
                    await ws.send('ping')
                    response = await ws.recv()
                    print('recv:', response)
                    await asyncio.sleep(5)
        except (ConnectionClosed, OSError):
            print(f'reconnect in {backoff}s')
            await asyncio.sleep(backoff)
            backoff = min(backoff * 2, 60)
```

</details>

---

## Ressources externes

- [websockets documentation](https://websockets.readthedocs.io/)
- [RFC 6455 - WebSocket Protocol](https://datatracker.ietf.org/doc/html/rfc6455)